<a href="https://colab.research.google.com/github/RezaDwiPuspita/Market-Basket-Analysis-Penjualan-/blob/main/Market_Basket_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit mlxtend

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 113.2 MB/s eta 0:00:00


In [ ]:
import streamlit as st
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules

# =========================
# MEMBACA DATA
# =========================
df = pd.read_csv("penjualancafefix.csv")

# =========================
# PREPROCESSING DATA
# =========================
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Time'] = pd.to_datetime(df['Time'], errors='coerce')

df['month'] = df['Date'].dt.month
df['day'] = df['Date'].dt.weekday
df['hour'] = df['Time'].dt.hour

# Mengubah angka bulan menjadi nama bulan
df['month'].replace(
    [1,2,3,4,5,6,7,8,9,10,11,12],
    ['Januari','Februari','Maret','April','Mei','Juni',
     'Juli','Agustus','September','Oktober','November','Desember'],
    inplace=True
)

# Mengubah angka hari menjadi nama hari
df['day'].replace(
    [0,1,2,3,4,5,6],
    ['Senin','Selasa','Rabu','Kamis',"Jum'at",'Sabtu','Minggu'],
    inplace=True
)

# Membuat kategori waktu
df['period_day'] = pd.cut(
    df['hour'],
    bins=[0,11,15,18,23],
    labels=['Pagi','Siang','Sore','Malam']
)

# weekday / weekend
df['weekday_weekend'] = df['day'].apply(
    lambda x: 'Weekend' if x in ['Sabtu','Minggu'] else 'Weekday'
)

# =========================
# JUDUL
# =========================
st.title("Market Basket Analysis Café")

# =========================
# INPUT USER
# =========================
item = st.selectbox("Items", df['Items'].unique())

period_day = st.selectbox(
    "Period Day",
    ['Pagi','Siang','Sore','Malam']
)

weekday_weekend = st.selectbox(
    "Weekday / Weekend",
    ['Weekday','Weekend']
)

month = st.selectbox(
    "Month",
    ['Januari','Februari','Maret','April','Mei','Juni',
     'Juli','Agustus','September','Oktober','November','Desember']
)

day = st.selectbox(
    "Day",
    ['Senin','Selasa','Rabu','Kamis',"Jum'at",'Sabtu','Minggu']
)

# =========================
# FILTER DATA
# =========================
data = df[
    (df['period_day'] == period_day) &
    (df['weekday_weekend'] == weekday_weekend) &
    (df['month'] == month) &
    (df['day'] == day)
]

# =========================
# JIKA DATA ADA
# =========================
if len(data) > 0:

    st.subheader("Filtered Data")
    st.dataframe(data)

    # GROUPING
    items_count = data.groupby(
        ['Gross Sales','Items']
    )['Items'].count().reset_index(name='Count')

    # PIVOT TABLE
    basket = items_count.pivot_table(
        index='Gross Sales',
        columns='Items',
        values='Count',
        aggfunc='sum'
    ).fillna(0)

    # ENCODING
    basket = basket.applymap(lambda x: 1 if x > 0 else 0)

    # =========================
    # APRIORI
    # =========================
    frequent_items = apriori(
        basket,
        min_support=0.05,
        use_colnames=True
    )

    st.subheader("Frequent Itemsets")
    st.dataframe(frequent_items)

    # =========================
    # ASSOCIATION RULES
    # =========================
    rules = association_rules(
        frequent_items,
        metric="confidence",
        min_threshold=0.3
    )

    st.subheader("Association Rules")
    st.dataframe(rules)

else:
    st.warning("Tidak ada data untuk filter tersebut")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag